# 02 - Baseline reproductible

Rejoue la baseline statistique de bout en bout depuis le manifeste d'ingestion et évalue ses sorties JSON sur l'échantillon RSNA. Le code réutilisable vit dans `src/` ; ce notebook ne fait qu'orchestrer.

> Prototype pédagogique. Non destiné au diagnostic.

In [ ]:
import os, sys, pathlib
root = pathlib.Path.cwd()
while not (root / 'pyproject.toml').exists() and root != root.parent:
    root = root.parent
os.chdir(root); sys.path.insert(0, str(root))
print('Racine du projet:', root)

In [ ]:
import json, tempfile
from src.inference import InferenceConfig, run_batch_inference

out = pathlib.Path(tempfile.mkdtemp()) / 'baseline'
result = run_batch_inference('data/manifests/ingest_manifest.csv', out, config=InferenceConfig())
print('Cas traités:', result.processed, '| rejetés:', result.rejected)

In [ ]:
# Exemple de sortie JSON conforme au contrat partagé
sample = sorted((out / 'cases').glob('*.json'))[0]
print(json.dumps(json.loads(sample.read_text())['prediction'], ensure_ascii=False, indent=2))

In [ ]:
from eval.metrics import evaluate_prediction_dir
ev = evaluate_prediction_dir(out)
for key in ('evaluated_cases', 'accuracy', 'macro_f1', 'macro_recall_sensitivity', 'macro_specificity', 'json_validity_rate', 'uncertainty_rate'):
    print(f'{key:26} {ev[key]}')
print('\nMatrice de confusion (réel -> prédit):')
print(json.dumps(ev['confusion_matrix'], ensure_ascii=False, indent=2))

La baseline atteint ~70% d'accuracy / macro-F1 sur les 30 cas. Sa faiblesse principale : deux opacités confondues avec `normal` (faux négatifs dangereux). Le notebook 03 mesure une amélioration ciblée sur ce point.